# Exploración del Espacio de Diseño Microarquitectónico (DSE) con Gem5 y McPAT

**Curso:** Arquitecturas Avanzadas de Computadores  
**Departamento de Ingeniería Electrónica — Universidad de Antioquia**  
**Profesor:** Sebastián Isaza  
**Estudiante:** Cristian Daniel Padrón Hernández  

## 1. Perfilado Inicial de la Aplicación (`jpeg2k_dec`)

### 1.1 Caracterización del Benchmark (`stats.txt` inicial)
En la simulación base del decodificador de imágenes **`jpeg2k_dec`** (sobre un ARM Cortex-A76 a $2.10\text{ GHz}$) se obtuvo un tiempo de ejecución de **$0.140693\text{ s}$**, con un **$\text{IPC} = 0.6853\text{ instrucciones/ciclo}$** y un **$\text{CPI} = 1.4592\text{ ciclos/instrucción}$**. Al revisar el archivo `stats.txt` (sobre $2.133 \times 10^8\text{ operaciones}$ completadas) se encontró lo siguiente:

* **Tipos de instrucciones:**
  * **Enteras (`IntAlu`):** Son el **$77.70\text{ \%}$** del programa ($1.658 \times 10^8\text{ operaciones}$).
  * **Memoria (`MemRead` y `MemWrite`):** Suman el **$21.75\text{ \%}$** ($14.55\text{ \%}$ de lecturas y $7.20\text{ \%}$ de escrituras, para un total de $4.640 \times 10^7\text{ operaciones}$).
  * **Vectoriales (`SIMD`):** Apenas llegan al **$0.54\text{ \%}$** ($1.149 \times 10^6\text{ operaciones}$), por lo que casi no pesan en la ejecución.
* **Cuellos de botella en Unidades Funcionales y Colas (`fuBusy` e `IQFull`):**
  * Hubo $1.152 \times 10^7\text{ intentos}$ detenidos porque la unidad funcional pedida estaba ocupada (`fuBusy`). De esos atascos, el **$78.56\text{ \%}$** ($9.052 \times 10^6\text{ paradas}$) ocurrió esperando una **ALU de enteros (`IntAlu`)** y el **$19.62\text{ \%}$** esperando puertos de memoria (`MemWrite`/`MemRead`).
  * Además, la etapa de renombrado se frenó **$1.557 \times 10^7\text{ veces}$ porque se llenó la cola de instrucciones (`IQFullEvents`)**, frente a solo **$3,871\text{ veces}$ por `ROB` lleno (`ROBFullEvents`)**.
* **Comportamiento inicial de las cachés (*Miss Rate*):**
  * **`L1i` (`icache`):** Tasa de fallos casi nula de **$0.0022\text{ \%}$** ($2,875\text{ fallos}$), lo que anticipa que cambiar el tamaño de `L1i` tendrá poco efecto.
  * **`L1d` (`dcache`) y `L2`:** La caché de datos `L1d` tuvo un **$0.94\text{ \%}$** de fallos ($493,033\text{ fallos}$) y la `L2` un **$48.67\text{ \%}$** de fallos locales ($189,391\text{ fallos}$).



In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15.5, 4.8), dpi=300)

# 1. Clases de instrucciones comprometidas (committedInstType)
clases = ["IntAlu\n(Enteros)", "MemRead\n(Lecturas)", "MemWrite\n(Escrituras)", "SIMD / Otros\n(Vectoriales)"]
valores_clases = [77.70, 14.55, 7.20, 0.55]
bars1 = ax1.bar(clases, valores_clases, color=["#1f77b4", "#2ca02c", "#98df8a", "#ff7f0e"], edgecolor="black", width=0.55)
ax1.set_title("Perfil de Instrucciones (stats.txt inicial)", fontweight="bold")
ax1.set_ylabel("Porcentaje de Operaciones (%)")
ax1.set_ylim(0, 90)
for b in bars1:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5, f"{b.get_height():.2f}%", ha="center", fontweight="bold")

# 2. Atascos por Unidades Funcionales ocupadas (statFuBusy)
fu_labels = ["IntAlu\n(ALU Entera)", "MemWrite\n(Escritura Mem)", "SimdCmp\n(Vectorial)", "MemRead\n(Lectura Mem)"]
fu_vals = [78.56, 18.82, 1.75, 0.80]
bars2 = ax2.bar(fu_labels, fu_vals, color=["#d62728", "#ff9896", "#9467bd", "#c5b0d5"], edgecolor="black", width=0.55)
ax2.set_title("Paradas por Unidad Funcional Ocupada (fuBusy)", fontweight="bold")
ax2.set_ylabel("Porcentaje sobre Total de Paradas FU (%)")
ax2.set_ylim(0, 90)
for b in bars2:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5, f"{b.get_height():.2f}%", ha="center", fontweight="bold")

# 3. Tasas de Fallo de Caché (Miss Rate %) en la corrida inicial
cache_labels = ["L1i (icache)\n128 kB", "L1d (dcache)\n64 kB", "L2 (l2cache)\n256 kB"]
miss_vals = [0.0022, 0.9402, 48.6749]
bars3 = ax3.bar(cache_labels, miss_vals, color=["#17becf", "#bcbd22", "#e377c2"], edgecolor="black", width=0.5)
ax3.set_title("Tasa de Fallos de Caché Inicial (Miss Rate)", fontweight="bold")
ax3.set_ylabel("Tasa de Fallos Local (%)")
ax3.set_ylim(0, 58)
for b in bars3:
    h = b.get_height()
    txt = f"{h:.4f}%" if h < 0.01 else f"{h:.2f}%"
    ax3.text(b.get_x() + b.get_width()/2, h + 1.2, txt, ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 2. Metodología de Exploración del Espacio de Diseño (DSE) y Algoritmo Genético

Para optimizar la microarquitectura del procesador, se definió un Espacio de Diseño (*Design Space*) de **1,728 combinaciones posibles** variando 6 parámetros clave de hardware:

- **Buffer de Reordenamiento (`ROB`):** 32, 64, 96, 128, 160, 192, 224, 256 entradas.
- **Cola de Instrucciones (`IQ`):** 16, 24, 32, 64 entradas.
- **Caché L2:** 256 kB, 512 kB, 1 MB, 2 MB.
- **Cachés L1 Inst. / L1 Datos:** 16 kB, 32 kB, 64 kB.
- **Ancho de Emisión (`Width`):** 2, 4, 6, 8 instrucciones/ciclo.

### 2.1 Heurística Evolutiva (Algoritmo Genético)
Se implementó un **Algoritmo Genético en Python** para explorar eficientemente el espacio, evaluando **1,002 configuraciones reales** a lo largo de **34 generaciones**:

1. **Población Inicial:** Generación de 25-30 individuos aleatorios en el espacio de búsqueda.
2. **Función de Aptitud (*Fitness*):** Evaluación del producto Energía-Retardo (**`EDP`** en Joules·segundo) obtenido mediante la salida combinada de **Gem5** y **McPAT**.
3. **Evolución:** Selección, cruce de parámetros de los mejores especímenes y mutaciones aleatorias.
4. **Convergencia:** A partir de la Generación 7 se alcanzó el óptimo global (**`EDP = 0.05666 J·s`**), manteniendo estabilidad en las generaciones subsecuentes (Gen 8 a 34).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style='whitegrid', palette='colorblind', font='DejaVu Sans')

# Cargar dataset
df = pd.read_csv('dse_1000_completo.csv')

# Graficar evolución por generaciones
gen_stats = df.groupby('Generacion').agg(
    min_EDP=('EDP_Js', 'min'),
    mean_EDP=('EDP_Js', 'mean'),
    max_IPC=('IPC', 'max'),
    mean_IPC=('IPC', 'mean')
).reset_index()
gen_stats['record_acumulado'] = gen_stats['min_EDP'].cummin()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(gen_stats['Generacion'], gen_stats['mean_EDP'], color='gray', linestyle=':', label='EDP Promedio Gen')
ax1.plot(gen_stats['Generacion'], gen_stats['min_EDP'], color='cornflowerblue', marker='o', label='Mínimo EDP Gen')
ax1.plot(gen_stats['Generacion'], gen_stats['record_acumulado'], color='darkblue', linewidth=2.5, label='Récord Acumulado')
ax1.set_title('Convergencia del Algoritmo Genético (34 Generaciones)', fontweight='bold')
ax1.set_ylabel('EDP (Joules · s)')
ax1.legend()

ax2.plot(gen_stats['Generacion'], gen_stats['max_IPC'], color='orange', marker='s', label='Máximo IPC Gen')
ax2.plot(gen_stats['Generacion'], gen_stats['mean_IPC'], color='darkorange', linestyle='--', label='Promedio IPC Gen')
ax2.set_xlabel('Generación del Algoritmo Genético')
ax2.set_ylabel('IPC')
ax2.legend()
sns.despine()
plt.tight_layout()
plt.show()


## 3. Resultados y Convergencia del Algoritmo Genético

Para analizar cómo se comportó el algoritmo a lo largo de las **34 generaciones**, se evaluó la evolución del producto energía-retardo (**EDP**) y del rendimiento (**IPC**). La gráfica superior muestra claramente las tres fases del proceso de optimización:

### 3.1 Comportamiento del EDP (Panel Superior)
* **Punteada gris (EDP Promedio):** Arranca alto en la Generación 1 (~0.061 Joules·s) debido a la exploración aleatoria de configuraciones sobredimensionadas. Al avanzar las generaciones, el promedio baja y se estabiliza alrededor de 0.058 - 0.059 J·s.
* **Línea azul (Mínimo por Generación):** Muestra el mejor individuo de cada tanda. Se ve cómo desciende rápidamente hasta encontrar el óptimo en la Generación 7.
* **Línea sólida azul oscuro (Récord Acumulado):** Refleja el mejor diseño histórico encontrado en toda la ejecución. Se aplana de forma definitiva en la **Generación 7 (`Exp #208`)** con un valor de **`0.05666 J·s`**, demostrando que el algoritmo encontró la zona ganadora muy temprano y la mantuvo bloqueada con éxito.

### 3.2 Comportamiento del IPC (Panel Inferior)
* **Línea sólida naranja (Máximo IPC):** Desde las primeras generaciones (específicamente en la **Gen 3, `Exp #70`**), el algoritmo localiza el pico máximo de rendimiento absoluto de todo el DSE (**`IPC = 0.7712`**), utilizando un pipeline ancho de 8 instrucciones y caché L2 de 2 MB.
* **Línea discontinua naranja (IPC Promedio):** Se mantiene muy estable a lo largo de las 34 generaciones oscilando entre **0.73 y 0.75**, lo que confirma que, aunque el algoritmo prioriza la eficiencia energética (el EDP), la población general logra retener un alto nivel de paralelismo sin colapsar el rendimiento.

## 3. Resultados y Configuraciones Campeonas

A partir de las 1,002 simulaciones ejecutadas con Gem5 y McPAT, se identificaron las tres configuraciones microarquitectónicas principales:

| Métrica / Parámetro |  Máximo Rendimiento |  Mínimo Consumo |  Óptimo EDP (Ganador) |
| :--- | :---: | :---: | :---: |
| **Experimento ID** | **Exp 70** (Gen 3) | **Exp 258** (Gen 9) | **Exp 208** (Gen 7) |
| **ROB / IQ** | 64 / 24 | 32 / 16 | **64 / 16** |
| **Caché L2** | 2 MB | 256 kB | **256 kB** |
| **Cachés L1i / L1d** | 64 kB / 64 kB | 16 kB / 32 kB | **64 kB / 64 kB** |
| **Ancho (Width)** | 8 | 4 | **4** |
| **IPC** | **0.7712** | 0.7353 | **0.7435** |
| **Tiempo de Ejecución** | **0.1250 s** | 0.1311 s | **0.1297 s** |
| **Potencia Total** | 3.8083 W | **3.3119 W** | **3.3697 W** |
| **Energía Total** | 0.4761 J | **0.4343 J** | **0.4370 J** |
| **EDP (Energy-Delay Product)** | 0.05953 J·s | 0.05695 J·s | **0.05666 J·s** |

### 3.1 Análisis Trade-Off (Frontera de Pareto)
* La arquitectura de **Óptimo EDP (Exp 208)** conserva el **96.4% del IPC máximo** (0.7435 frente a 0.7712), reduciendo la **energía total en un 8.2%** y la **potencia estática por fuga en un 20.3%**.
* Se demuestra empíricamente que estructuras de hardware sobredimensionadas (L2 de 2 MB, Width de 8) no se justifican para esta carga de trabajo debido a la saturación previa en la ALU de enteros.


In [ ]:
# Generar Frontera de Pareto (IPC vs. Energía)
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(df['Energia_J'], df['IPC'], c=df['EDP_Js'], cmap='viridis_r', alpha=0.6, s=35)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('EDP (Joules · s) [Menor es mejor]')

# Curva de Pareto
sorted_df = df.sort_values(by=['Energia_J', 'IPC'], ascending=[True, False])
pareto_front = []
max_ipc_so_far = -1
for _, row in sorted_df.iterrows():
    if row['IPC'] > max_ipc_so_far:
        pareto_front.append(row)
        max_ipc_so_far = row['IPC']
pareto_df = pd.DataFrame(pareto_front)

idx_max_ipc = df['IPC'].idxmax()
idx_min_energia = df['Energia_J'].idxmin()
idx_best_edp = df['EDP_Js'].idxmin()

ax.plot(pareto_df['Energia_J'], pareto_df['IPC'], color='crimson', linestyle='--', linewidth=2, label='Frontera de Pareto')
ax.scatter(df.loc[idx_max_ipc, 'Energia_J'], df.loc[idx_max_ipc, 'IPC'], color='gold', s=160, edgecolors='black', label='Máx IPC (Exp 70)')
ax.scatter(df.loc[idx_min_energia, 'Energia_J'], df.loc[idx_min_energia, 'IPC'], color='mediumseagreen', s=160, edgecolors='black', label='Mín Energía (Exp 258)')
ax.scatter(df.loc[idx_best_edp, 'Energia_J'], df.loc[idx_best_edp, 'IPC'], color='dodgerblue', s=180, marker='*', edgecolors='black', label='Óptimo EDP (Exp 208)')

ax.set_title('Espacio de Diseño DSE (1,002 Simulaciones): IPC vs. Energía', fontweight='bold', pad=15)
ax.set_xlabel('Energía Total Consumida (Joules) [Menor es mejor]')
ax.set_ylabel('Rendimiento (IPC) [Mayor es mejor]')
ax.legend(loc='lower right')
sns.despine()
plt.tight_layout()
plt.show()


### 3.2 Análisis del Espacio de Diseño y Frontera de Pareto (IPC vs. Energía)

La gráfica anterior mapea los **1,002 experimentos exitosos** ubicando el rendimiento en el eje vertical ($\text{IPC}$, donde mayor es mejor) frente a la energía total en el eje horizontal ($\text{Joules}$, donde menor es mejor). Los puntos están coloreados según su **EDP** (con la barra lateral indicando que los tonos más amarillos representan la mejor eficiencia y los morados la peor). Del análisis se destacan tres puntos críticos:

1. **Frontera de Pareto (Línea discontinua roja):** Delimita el conjunto de configuraciones óptimas donde no es posible mejorar el rendimiento sin incrementar el consumo energético (o viceversa).
2. **Máximo Rendimiento (`Exp #70`, círculo amarillo superior):** Alcanza un $\text{IPC}$ de **$0.7712$** con un consumo de $0.4761\text{ J}$. Se ubica en el extremo derecho de la frontera porque requiere estructuras grandes (`Width = 8` y `L2 = 2MB`), lo que incrementa notablemente la energía.
3. **Mínima Energía (`Exp #258`, círculo verde inferior izquierdo):** Consume la menor cantidad de energía de todo el DSE (**$0.4343\text{ J}$**) con un $\text{IPC}$ de $0.7353$, gracias a su configuración compacta (`Width = 4`, `ROB = 32`, `L2 = 256kB`).
4. **Óptimo EDP / El Ganador (`Exp #208`, estrella azul):** Representa el mejor balance (*sweet spot*). Logra un $\text{IPC}$ de **$0.7435$** (conservando el **96.4 %** del rendimiento máximo) con una energía muy cercana al mínimo (`0.4370 J`), obteniendo el menor producto energía-retardo de todo el proyecto (**$0.05666\text{ J}\cdot\text{s}$**).

## 4. Tiempos de Simulación en Gem5 y McPAT

| Simulador | Promedio 1 Sim. (min) | Total 1,002 Sims. (Horas) | Total (Días CPU) | Porcentaje |
| :--- | :---: | :---: | :---: | :---: |
| **Gem5** | 24.40 min | 407.44 h | 16.98 días | 97.47 % |
| **McPAT** | 0.63 min | 10.56 h | 0.44 días | 2.53 % |
| **Total Acumulado** | **25.03 min** | **418.01 h** | **17.42 días** | **100.00 %** |

### 4.1 Análisis del Esfuerzo Computacional
* **Tiempo Acumulado de CPU:** 418.01 horas (17.42 días) si se hubieran simulado de forma estrictamente secuencial en un único núcleo.
* **Tiempo Real de Ejecución:** Gracias a la automatización paralela en **10 hilos de trabajo**, la exploración completa tomó **~41.8 horas reales** (~1.74 días).


## 5. Conclusiones y Recomendaciones de Diseño

1. **Cuello de Botella Dominante:** El perfilado reveló que `jpeg2k_dec` está fuertemente limitado por cómputo entero (77.7% de operaciones `IntAlu` y 78.56% de paradas en ALU). Por esta razón, ampliar el pipeline a 8 instrucciones/ciclo o escalar el ROB por encima de 64 no se traduce en ganancias significativas de rendimiento.
2. **Arquitectura Ganadora:** La configuración de **Óptimo EDP (Exp 208: ROB=64, IQ=16, L2=256kB, Width=4)** representa el diseño más eficiente para este benchmark. Logra un IPC de 0.7435 (casi idéntico al máximo) recortando la potencia estática de fuga en un 20.3% respecto a la configuración sobredimensionada.
3. **Eficiencia Metodológica**: La integración de un algoritmo genético permitió resolver el problema de optimización microarquitectónica en tan solo 7 generaciones de convergencia efectiva, reduciendo el espacio de búsqueda en un $93.88\text{%}$ y validando la viabilidad del diseño.

#6.  Datos Crudos y Resultados Completos de Simulación
Los resultados y archivos de salida brutos generados por Gem5 y McPAT para los 1,002 experimentos se encuentran disponibles en el siguiente enlace: https://drive.google.com/drive/folders/1jQFs0_Kg1AypuDafpum4CjjWmDOtQjIL?usp=sharing